> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# Notebook 05 | Tourism Accessibility and Spatial Hotel Profiles
## KBS | Luís Silvano | Portuguese Hotel Dataset (2018–2024)

**Research question:** *How do local tourism accessibility profiles differ across independent hotels, and how do these spatial profiles relate descriptively to hotel characteristics and labour productivity?*

This notebook transforms **hotel × point-of-interest (POI) matrices** into hotel-level accessibility indicators; identifies **spatial-context clusters without using productivity to form the clusters**; and generates interactive maps and publication-quality figures.

**Data sources:** the main 2018–2024 Excel dataset and geographical tables for castles, churches, beaches, and restaurants/cafés, where available. The supplied package includes the two Excel files that were provided and a CSV containing the relevant columns extracted from the main Excel dataset. Variables already available in the main Excel file serve as *proxies*, and the notebook explicitly distinguishes their provenance.

**Important:** A hotel–POI *cross join* is not a temporal panel. Each hotel receives one spatial feature vector, which may subsequently be linked to the **hotel × year** panel.


## 0. How to Run This Notebook

1. Extract the entire ZIP archive into a single folder. Keep the `.ipynb`, both `.xlsx` files, `hoteis_2018_2024_subset.csv`, and `manifesto_origem.json` **together in that folder**.
2. Launch Jupyter in that folder. If dependencies are missing, run `pip install -r requirements_KBS05.txt` in the same Python environment used by Jupyter.
3. Run **Kernel > Restart & Run All**. HTML maps will be saved to `resultados_espaciais_05` and can be opened in a browser.
4. When Luís provides the Excel files for beaches, churches, and restaurants/cafés, place them in the same folder under their original filenames. The notebook detects them automatically. For the (very large) restaurants/cafés file, enable the dedicated option for full-file reading.

**Key concepts:** POI = point of interest; Haversine = distance along the Earth's surface; KNN = the `k` nearest POIs to each hotel (not to be confused with KNN regression); cluster = a group of hotels with similar spatial profiles.


In [ ]:
from pathlib import Path
import sys, json, math, re, html, hashlib, heapq, warnings
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

try:
    import folium
    from branca.colormap import linear
    FOLIUM_OK = True
except ImportError:
    FOLIUM_OK = False
    print('WARNING: folium/branca are not installed. Static maps will work, but interactive HTML maps will be unavailable.')

try:
    from lxml import etree as ET
    XML_RAPIDO = True
except ImportError:
    XML_RAPIDO = False

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi':120, 'savefig.dpi':240, 'axes.titlesize':13,
                     'axes.labelsize':10, 'font.size':10})
SEMENTE = 42
K_MAX = 5
D0_KM = 1.0
RAIOS_KM = [0.8, 2.0, 5.0, 10.0]
APENAS_INDEPENDENTES = True
LER_MESTRE_DIRETO_DO_EXCEL = False   # True: skip the curated CSV and reread the raw Excel file
LER_RESTAURANTES_COMPLETO = False    # True: full parsing of a >600 MB file can take a long time
MOSTRAR_RESULTADOS = True
ROOT = Path.cwd()
OUT = ROOT/'resultados_espaciais_05'
OUT.mkdir(parents=True, exist_ok=True)
CORES = ['#0072B2','#D55E00','#009E73','#CC79A7','#E69F00','#56B4E9','#7A5195']

print('Working directory:',ROOT.resolve())
print('Outputs:',OUT.resolve())

In [ ]:
# Verify that the required files are in the same folder as the notebook
ARQ_MESTRE = ROOT/'2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx'
ARQ_CURADO = ROOT/'hoteis_2018_2024_subset.csv'
ARQ_MANIFESTO = ROOT/'manifesto_origem.json'
PADROES = {
 'castelos':['*Castles*Distances*.xlsx','*Castles*.xlsx'],
 'igrejas':['*Churches*Distances*.xlsx','*Churches*.xlsx'],
 'praias':['*Beaches*Distances*.xlsx','*Beaches*.xlsx'],
 'restaurantes':['*Restaurants*Coffes*.xlsx','*Restaurants*Coffee*.xlsx']
}
def locate_file(padroes):
    # Case-insensitive filename matching for Windows file names
    import fnmatch
    cand = sorted([p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower()=='.xlsx'])
    for padrao in padroes:
        matches=[p for p in cand if fnmatch.fnmatch(p.name.casefold(),padrao.casefold())]
        if matches: return matches[0]
    return None

ARQUIVOS_POI = {cat:locate_file(pats) for cat,pats in PADROES.items()}
if not ARQ_CURADO.exists() and not ARQ_MESTRE.exists():
    raise FileNotFoundError('Place the main Excel file or the curated CSV in this folder.')

inspecao = pd.DataFrame([
 {'categoria':c,'arquivo':p.name if p else 'MISSING',
  'tamanho_MB':round(p.stat().st_size/1024**2,1) if p else np.nan}
 for c,p in ARQUIVOS_POI.items()
])
display(inspecao)
print('Main Excel file present:',ARQ_MESTRE.exists(),'| Supporting CSV present:',ARQ_CURADO.exists())
if ARQUIVOS_POI['castelos'] is None:
    print('WARNING: castle POI data are missing. The analysis will use only the existing indicator in the main Excel dataset.')
if ARQUIVOS_POI['igrejas'] is None or ARQUIVOS_POI['praias'] is None:
    print('WARNING: church/beach indicators are proxied using the main dataset; complete K5 indicators cannot be calculated for these categories.')

## 1. Mathematical Definitions and Good Practices

For hotel $i$ and category $c$ (castles, churches, beaches, or restaurants), let $d^{(c)}_{i,(1)} \leq \cdots \leq d^{(c)}_{i,(k)}$ denote the distances to the $k$ nearest points of interest, in kilometres.

**1. Geodesic distance (Haversine):**

$$\Delta \varphi = \varphi_2-\varphi_1,\quad \Delta\lambda=\lambda_2-\lambda_1,\qquad a=\sin^2\!\left(\frac{\Delta\varphi}{2}\right)+\cos\varphi_1\cos\varphi_2\sin^2\!\left(\frac{\Delta\lambda}{2}\right),$$
$$d=2R\arcsin(\sqrt{a}),\qquad R=6371.0088\ \mathrm{km}.$$

**2. Accessibility index using $k$ neighbours** (a dimensionally consistent version of the original idea):

$$A_i^{(c)}(k;d_0)=\frac{1}{k}\sum_{j=1}^k \frac{1}{1+d_{i,(j)}^{(c)}/d_0},\qquad d_0=1\ \mathrm{km}.$$

Properties: $0<A\leq 1$; higher values indicate closer POIs. Dividing by $d_0$ prevents a dimensionless number from being added directly to a distance. We compute $k=1,3,5$ whenever a complete matrix is available for the category.

**3. Counts and density within radius $r$:**

$$N_i^{(c)}(r)=\sum_{p\in\mathcal P_c}\mathbb{1}\{d(i,p)\le r\},\qquad D_i^{(c)}(r)=\frac{N_i^{(c)}(r)}{\pi r^2}.$$

**Note:** $D$ is an operational POI density per km$^2$ within an approximate geodesic circle, not an estimate corrected for coastline, population, or land area. The count of restaurants/cafés within 800 m indicates **nearby services**, not evidence of the establishments' quality.

**4. Clustering:** We use a four-dimensional spatial feature matrix (without productivity):

$$\mathbf{x}_i=\bigl[A_i^{(castelos)}(1),A_i^{(igrejas)}(1),A_i^{(praias)}(1),\log(1+D_i^{(rest.)}(0.8))\bigr].$$

After median imputation, exploratory outlier control, and standardisation, K-Means minimises

$$\min_{\mathcal C_1,\dots,\mathcal C_K}\sum_{k=1}^{K}\sum_{i\in\mathcal C_k}\|\mathbf{z}_i-\boldsymbol\mu_k\|^2.$$

The number $K$ is explored using the *silhouette score* and stability across random seeds. Cluster profiles are named **post hoc** based on the medians, never based on productivity. The target variable is analysed only after clustering.

**Provenance:** If complete geographic matrices are unavailable, beach/church $A(1)$ is computed from the nearest-distance values already supplied in the main Excel dataset. Their distance metric, collection year, and comparability must be confirmed with the data author before submission. We never estimate $A(5)$ from a single distance.


In [ ]:
# 1.1 Fast, streaming read of large Excel files: memory is limited to the requested columns.
# This implementation supports cross joins containing many rows.
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import zipfile

def validate_xlsx_header(caminho):
    pasta = load_workbook(caminho,read_only=True,data_only=True)
    if len(pasta.sheetnames)>1:
        print(f'WARNING {caminho.name}: the first worksheet will be used {pasta.sheetnames[0]!r}.')
    folha = pasta.worksheets[0]
    nomes=[str(v).strip() if v is not None else ''
           for v in next(folha.iter_rows(min_row=1,max_row=1,values_only=True))]
    pasta.close()
    return nomes

def iterate_xlsx_rows(caminho, colunas):
    """Yield only requested columns without loading the entire cross join."""
    cabecalho=validate_xlsx_header(caminho)
    faltas=[c for c in colunas if c not in cabecalho]
    if faltas: raise ValueError(f'{caminho.name}: missing columns: {faltas}; available: {cabecalho[:30]}')
    if not XML_RAPIDO:
        # Slower but functional fallback. For very large files, install lxml.
        wb=load_workbook(caminho,read_only=True,data_only=True)
        ws=wb.worksheets[0]
        indices={c:cabecalho.index(c) for c in colunas}
        for row in ws.iter_rows(min_row=2,values_only=True):
            yield {c:row[i] for c,i in indices.items()}
        wb.close()
        return
    ns='{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
    letras={get_column_letter(i+1):c for i,c in enumerate(cabecalho) if c in colunas}
    with zipfile.ZipFile(caminho) as arq:
        strings=[]
        if 'xl/sharedStrings.xml' in arq.namelist():
            raiz=ET.parse(arq.open('xl/sharedStrings.xml')).getroot()
            strings=[''.join(si.itertext()) for si in raiz.findall(ns+'si')]
            del raiz
        with arq.open('xl/worksheets/sheet1.xml') as stream:
            parser=ET.iterparse(stream,events=('end',),tag=ns+'row',huge_tree=True)
            for _,row in parser:
                if row.get('r')!='1':
                    item={}
                    for cel in row.iterchildren(tag=ns+'c'):
                        nome=letras.get(cel.get('r','').rstrip('0123456789'))
                        if nome is None: continue
                        tp=cel.get('t')
                        valor=cel.findtext(ns+'v')
                        if valor is None and tp=='inlineStr':
                            is_node=cel.find(ns+'is')
                            valor=''.join(is_node.itertext()) if is_node is not None else None
                        if valor is None: continue
                        if tp=='s': valor=strings[int(valor)]
                        elif tp=='b': valor=(valor=='1')
                        elif tp not in ('str','inlineStr','e'):
                            try: valor=float(valor)
                            except (TypeError,ValueError): pass
                        item[nome]=valor
                    yield item
                row.clear()
                while row.getprevious() is not None:
                    del row.getparent()[0]

def normalize_id(valor):
    if valor is None or pd.isna(valor): return None
    s=str(valor).strip()
    if re.fullmatch(r'[+-]?\d+(?:\.0+)?',s):
        return str(int(float(s)))
    return s or None

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
# The main dataset contains hotels from multiple years. Create ONE row per hotel
# for spatial clustering; keep the panel separately for year-by-year analyses.
COLS_MESTRE = [
 'ht_id','ht_registration_number','f_year','ht_hotel_chain_d','ht_name',
 'ht_latitude','ht_longitude','ht_municipality','ht_nuts_3_newcriteria',
 'ht_nuts_2_newcriteria','ht_accommodation_type','ht_quality_stars',
 'ht_capacity','ht_number_of_accommodation_units','ht_location_beach1',
 'ht_location_church1','ht_location_castle1','ht_Restaurants&CoffesL800m',
 'f_labprodct','f_number_of_employees','f_operating_revenue_turnover2',
 'ht_spa_d','ht_gym_d','ht_outdoor_pools_d','ht_meeting_rooms_d'
]

cache_ok = False
if ARQ_CURADO.exists() and ARQ_MANIFESTO.exists() and not LER_MESTRE_DIRETO_DO_EXCEL:
    mani=json.loads(ARQ_MANIFESTO.read_text(encoding='utf-8'))
    cache_ok = (
        (not ARQ_MESTRE.exists() or file_sha256(ARQ_MESTRE)==mani.get('base_mestre_sha256'))
        and file_sha256(ARQ_CURADO)==mani.get('csv_curado_sha256')
        and set(COLS_MESTRE).issubset(pd.read_csv(ARQ_CURADO,nrows=0).columns)
    )
if cache_ok:
    painel=pd.read_csv(ARQ_CURADO,low_memory=False)
    print('Fast import: curated CSV verified against the original Excel file (SHA-256).')
else:
    if not ARQ_MESTRE.exists():
        raise FileNotFoundError('Curated CSV is missing/invalid, and the main Excel file was not found.')
    painel=pd.DataFrame(iterate_xlsx_rows(ARQ_MESTRE,COLS_MESTRE),columns=COLS_MESTRE)
    painel.to_csv(OUT/'subset_extraido_do_excel_atual.csv',index=False)
    print('Fallback import: data extracted directly from the updated Excel file.')

for col in COLS_MESTRE:
    if col not in painel.columns: raise ValueError('Missing required column: '+col)
for col in COLS_MESTRE:
    if col not in ['ht_name','ht_municipality','ht_nuts_3_newcriteria','ht_nuts_2_newcriteria','ht_accommodation_type']:
        painel[col]=pd.to_numeric(painel[col],errors='coerce')
painel['hotel_key']=painel['ht_registration_number'].map(normalize_id)
painel=painel[painel['f_year'].between(2018,2024)].copy()
if APENAS_INDEPENDENTES:
    painel=painel[painel['ht_hotel_chain_d'].eq(0)].copy()
painel=painel.dropna(subset=['hotel_key'])
painel=painel.sort_values(['hotel_key','f_year'])
if painel.duplicated(['hotel_key','f_year']).any():
    print('WARNING: duplicate hotel/year pairs detected; only the first occurrence will be retained.')
    painel=painel.drop_duplicates(['hotel_key','f_year'],keep='first')

hoteis=painel.sort_values('f_year').drop_duplicates('hotel_key',keep='last').copy()
hoteis=hoteis.drop(columns=['f_labprodct'],errors='ignore')
hoteis=hoteis[hoteis['ht_latitude'].between(-90,90) & hoteis['ht_longitude'].between(-180,180)]
hoteis=hoteis[(hoteis['ht_latitude'].abs()>1)&(hoteis['ht_longitude'].abs()>1)].copy()
print(f'Panel: {len(painel):,} rows, {painel.hotel_key.nunique():,} hotels; spatial sample: {len(hoteis):,} hotels.')
print('Years:',painel.f_year.value_counts().sort_index().to_dict())
print('Percentage with valid productivity:',round(100*painel.f_labprodct.notna().mean(),1))
display(hoteis[['hotel_key','ht_municipality','ht_latitude','ht_longitude','ht_capacity']].head(5))

## 2. Extraction and Validation of Hotel–POI Distances

**We do not treat `EuclideanDistancekm` as a geographic distance without validation.** For latitude/longitude coordinates, Euclidean distance measured in degrees is not equivalent to kilometres along the Earth's surface. The notebook recomputes Haversine distances from coordinates and exports a comparative audit.

To avoid loading all pairs into memory, we process each file row by row and retain only the five shortest distances and counts within specified radii for each hotel. Memory requirements scale with the number of hotels, not with the size of the *cross join*.


In [ ]:
RAIO_TERRA_KM=6371.0088

def haversine_km(lat1,lon1,lat2,lon2):
    v1,v2=np.radians([lat1,lat2]); dl=np.radians(lon2-lon1)
    dphi=v2-v1
    a=np.sin(dphi/2)**2 + np.cos(v1)*np.cos(v2)*np.sin(dl/2)**2
    return 2*RAIO_TERRA_KM*np.arcsin(np.sqrt(np.clip(a,0,1)))

def find_matching_column(nomes,preferidas,fragmentos=None,excluidos=None):
    ex=set(excluidos or [])
    for p in preferidas:
        for c in nomes:
            if c not in ex and c.casefold()==p.casefold():return c
    for pedaco in fragmentos or []:
        for c in nomes:
            if c not in ex and pedaco.casefold() in c.casefold():return c
    return None

def extract_poi_indicators(caminho,categoria,k_max=5):
    """Extract nearest POIs and radius counts from hotel–POI pairs without a huge DataFrame."""
    cab=validate_xlsx_header(caminho)
    idc=find_matching_column(cab,['Numero do Registo','ht_registration_number','Registration Number'],['registro','registo','registration'])
    hlat=find_matching_column(cab,['ht_latitude','hotel_latitude'],['ht_lat','hotel.lat'])
    hlon=find_matching_column(cab,['ht_longitude','hotel_longitude'],['ht_lon','hotel.lon'])
    plat=find_matching_column(cab,['TouristicPoints.tp_Latitude','tp_Latitude'],['tp_latitude','point_latitude'],excluidos=[hlat])
    plon=find_matching_column(cab,['TouristicPoints.tp_Longitude','tp_Longitude'],['tp_longitude','point_longitude'],excluidos=[hlon])
    nome=find_matching_column(cab,['TouristicPoints.tp_Categorie_Name','tp_Name'],['categorie_name','point_name'])
    antigo=find_matching_column(cab,['EuclideanDistancekm'],['distancekm'])
    faltam=[x for x,n in [('hotel ID',idc),('hotel lat',hlat),('hotel lon',hlon),('POI lat',plat),('POI lon',plon)] if n is None]
    if faltam: raise ValueError(f'Unrecognised schema in {caminho.name}: {faltam}. Columns: {cab}')
    cols=list(dict.fromkeys([x for x in (idc,hlat,hlon,plat,plon,nome,antigo) if x]))
    grupos={}
    n_linhas=0; invalidas=0; comparacoes=[]; todos_pontos=set()
    for linha in iterate_xlsx_rows(caminho,cols):
        n_linhas += 1
        key=normalize_id(linha.get(idc))
        if key is None:
            invalidas += 1; continue
        try:
            a,b,c,d=[float(linha.get(x)) for x in (hlat,hlon,plat,plon)]
            if not (-90<=a<=90 and -90<=c<=90 and -180<=b<=180 and -180<=d<=180):
                raise ValueError('invalid coordinates')
            if (a==0 and b==0) or (c==0 and d==0):
                raise ValueError('null coordinates')
            dist=float(haversine_km(a,b,c,d))
        except (TypeError,ValueError,OverflowError):
            invalidas += 1; continue
        if nome:
            todos_pontos.add((str(linha.get(nome)),round(c,6),round(d,6)))
        if antigo and len(comparacoes)<3000:
            try:
                dant=float(linha.get(antigo))
                if np.isfinite(dant): comparacoes.append((dist,dant))
            except (TypeError,ValueError):pass
        g=grupos.setdefault(key,{'heap':[],'n':0,'radii':Counter()})
        g['n'] += 1
        h=g['heap']
        if len(h)<k_max:heapq.heappush(h,-dist)
        elif dist < -h[0]:heapq.heapreplace(h,-dist)
        for r in RAIOS_KM:
            if dist<=r:g['radii'][r] += 1
    itens=[]
    for key,g in grupos.items():
        distancias=sorted([-x for x in g['heap']])
        item={'hotel_key':key,'categoria':categoria,'n_pontos_encontrados':g['n']}
        for k in (1,3,5):
            if len(distancias)>=k:
                ds=np.array(distancias[:k],dtype=float)
                item[f'd_{categoria}_media{k}_km']=ds.mean()
                item[f'a_{categoria}_k{k}']=np.mean(1.0/(1.0+ds/D0_KM))
            else:
                item[f'd_{categoria}_media{k}_km']=np.nan
                item[f'a_{categoria}_k{k}']=np.nan
        for r in RAIOS_KM:
            tag=str(r).replace('.','p')
            item[f'n_{categoria}_r{tag}km']=g['radii'][r]
            item[f'dens_{categoria}_r{tag}km']=g['radii'][r]/(math.pi*r*r)
        itens.append(item)
    if todos_pontos:
        pois=pd.DataFrame(list(todos_pontos),columns=['nome','latitude','longitude'])
        pois=pois.sort_values(['nome','latitude','longitude'])
        pois.to_csv(OUT/f'13_pontos_{categoria}.csv',index=False)
    resumo={'categoria':categoria,'ficheiro':caminho.name,'linhas':n_linhas,
            'pares_invalidos':invalidas,'hoteis':len(grupos),'pontos_distintos_aprox':len(todos_pontos)}
    if comparacoes:
        dif=np.array(comparacoes,dtype=float)
        resumo['mediana_erro_absoluto_distancia_original_km']=float(np.median(abs(dif[:,0]-dif[:,1])))
        resumo['mediana_erro_relativo_original']=float(np.median(abs(dif[:,0]-dif[:,1])/(dif[:,0]+1e-6)))
    return pd.DataFrame(itens),resumo

# Sanity checks: identical coordinates yield 0 km; Lisbon–Porto is ~275 km in a straight line
assert abs(haversine_km(38.7223,-9.1393,38.7223,-9.1393))<1e-8
assert 265 < haversine_km(38.7223,-9.1393,41.1579,-8.6291) < 290
print('Haversine distance: geometry sanity check passed.')

In [ ]:
# If a POI category is missing, do not invent neighbours: only use the distance
# to the NEAREST known point from the master file, recording its provenance.
mapa_proxy={'castelos':'ht_location_castle1','igrejas':'ht_location_church1','praias':'ht_location_beach1'}
base=hoteis.copy()
auditoria=[]
fontes=[]

for categoria in ('castelos','igrejas','praias','restaurantes'):
    caminho=ARQUIVOS_POI.get(categoria)
    usar_matriz=(caminho is not None and (categoria!='restaurantes' or LER_RESTAURANTES_COMPLETO))
    if usar_matriz:
        print('Processing cross join for',categoria,':',caminho.name)
        feat,desc=extract_poi_indicators(caminho,categoria)
        if feat.empty:
            print('No valid pairs in',caminho.name,'; the available secondary source will be used.')
            usar_matriz=False
        else:
            base=base.merge(feat.drop(columns=['categoria']),how='left',on='hotel_key',validate='one_to_one')
            auditoria.append(desc)
            fontes.append({'categoria':categoria,'fonte':'cross_join_geodesico','ficheiro':caminho.name})
            print('  Hotels with matched pairs:',desc['hoteis'],'| Distinct POIs (approx.):',desc['pontos_distintos_aprox'])
            if 'mediana_erro_absoluto_distancia_original_km' in desc:
                print('  Median difference between supplied distances and Haversine (km):',
                      round(desc['mediana_erro_absoluto_distancia_original_km'],3))
    if not usar_matriz:
        if categoria in mapa_proxy:
            col=mapa_proxy[categoria]
            d=pd.to_numeric(base[col],errors='coerce').where(lambda x:x>=0)
            base[f'd_{categoria}_media1_km']=d
            base[f'a_{categoria}_k1']=1.0/(1.0+d/D0_KM)
            base[f'a_{categoria}_k3']=np.nan
            base[f'a_{categoria}_k5']=np.nan
            fontes.append({'categoria':categoria,'fonte':'distancia_mais_proxima_mestre_METRICA_PENDENTE','ficheiro':ARQ_MESTRE.name})
        else:
            fontes.append({'categoria':categoria,'fonte':'contagem_800m_mestre','ficheiro':ARQ_MESTRE.name})

# Retain the restaurant/café count within 800 m in all versions of the dataset.
contagem_master=pd.to_numeric(base['ht_Restaurants&CoffesL800m'],errors='coerce').clip(lower=0)
if 'n_restaurantes_r0p8km' not in base.columns:
    base['n_restaurantes_r0p8km']=contagem_master
else:
    base['n_restaurantes_r0p8km']=base['n_restaurantes_r0p8km'].fillna(contagem_master)
base['dens_restaurantes_r0p8km']=base['n_restaurantes_r0p8km']/(math.pi*0.8**2)
base['log_dens_restaurantes_r0p8km']=np.log1p(base['dens_restaurantes_r0p8km'])

pd.DataFrame(fontes).to_csv(OUT/'01_proveniencia_indicadores.csv',index=False)
if auditoria:
    pd.DataFrame(auditoria).to_csv(OUT/'02_auditoria_distancias_geograficas.csv',index=False)
display(pd.DataFrame(fontes))
print('Note: k=3 and k=5 are NaN by definition for categories without a complete matrix.')

## 3. Indicators and Descriptive Statistics

The hotel-clustering feature vector should comprise four dimensions that are comparable across categories: accessibility to the nearest castle, nearest church, and nearest beach, plus the density of restaurants/cafés within 800 m.

We automatically use $k=1$ for clustering **until all complete matrices are available**. Once full cross joins for castles, churches, and beaches, each with five POIs per hotel, are available, the same code switches to $k=5$ for all three categories. This avoids comparing a category represented by five neighbours against another with only its nearest point. We export $A(3)$ and $A(5)$ wherever data coverage permits, for subsequent sensitivity analyses.


In [ ]:
# Use the same k for all three attraction categories.
# Use k=5 when all three cross joins are complete; otherwise, use k=1.
PREFERIR_K5_NO_CLUSTER = True
CATEGORIAS_PONTUAIS = ('castelos','igrejas','praias')
completude_k5 = all((f'a_{c}_k5' in base and base[f'a_{c}_k5'].notna().mean()>=0.95)
                   for c in CATEGORIAS_PONTUAIS)
K_ACESSO_CLUSTER = 5 if PREFERIR_K5_NO_CLUSTER and completude_k5 else 1
DIMENSOES={
  f'a_castelos_k{K_ACESSO_CLUSTER}':f'Castle accessibility (k={K_ACESSO_CLUSTER})',
  f'a_igrejas_k{K_ACESSO_CLUSTER}':f'Church accessibility (k={K_ACESSO_CLUSTER})',
  f'a_praias_k{K_ACESSO_CLUSTER}':f'Beach accessibility (k={K_ACESSO_CLUSTER})',
  'log_dens_restaurantes_r0p8km':'Restaurant/café density (log1p)'
}
print('k used in the clustering feature matrix:',K_ACESSO_CLUSTER)
if K_ACESSO_CLUSTER==1:
    print('Reason: not all complete matrices are present; k=1 avoids mixing definitions. '
          'Once the church/beach Excel files are provided, k=5 can be used.')
NOMES=list(DIMENSOES.keys())
fonte_ok={cat: next((x['fonte'] for x in fontes if x['categoria']==cat),'na') for cat in ['castelos','igrejas','praias','restaurantes']}
base['n_dimensoes_validas']=base[NOMES].notna().sum(axis=1)
base=base[base['n_dimensoes_validas']>=3].copy()

print('Hotels with at least 3 valid spatial dimensions:',len(base))
display(base[NOMES].describe(percentiles=[0.1,0.5,0.9]).T.round(3))
print('Missing proportion by feature:')
display(base[NOMES].isna().mean().round(3).to_frame('fracao_na'))

fig,axs=plt.subplots(2,2,figsize=(11,7))
for ax,(col,nome) in zip(axs.flat,DIMENSOES.items()):
    x=base[col].dropna().to_numpy()
    ax.hist(x,bins=32,color='#4472A4',alpha=.83,edgecolor='white')
    ax.set_title(nome)
    ax.set_ylabel('Number of hotels')
    ax.set_xlabel('Dimensionless index (or log-density)')
plt.tight_layout()
plt.savefig(OUT/'figura_01_distribuicoes_dimensoes.png',bbox_inches='tight')
plt.show()

## 4. Selecting the Number of Clusters and Assessing Stability

We select $K$ from 2 to 6 using the *silhouette score* while examining stability across alternative random seeds (Adjusted Rand Index, ARI). This **does not prove** that the number of clusters corresponds to natural categories; it is an exploratory analysis with a degree of robustness checking. A cluster is not a ground-truth class.

Missing values are imputed using medians; spatial feature values are clipped to the 1st and 99th percentiles **only for clustering** (without excluding hotels or modifying productivity). Features are standardised to prevent the restaurant index from dominating merely because it uses a different scale.


In [ ]:
from sklearn.impute import SimpleImputer

amostra=base[NOMES].copy()
imputador=SimpleImputer(strategy='median')
mat=np.asarray(imputador.fit_transform(amostra),dtype=float)
q01=np.nanquantile(mat,.01,axis=0)
q99=np.nanquantile(mat,.99,axis=0)
mat_clip=np.clip(mat,q01,q99)
escaler=StandardScaler()
Z=escaler.fit_transform(mat_clip)

# Evaluate K on the same set of hotels, without productivity.
ks=list(range(2,min(7,len(Z)-1)))
linhas=[]
for k in ks:
    modelo=KMeans(n_clusters=k,n_init=15,random_state=SEMENTE)
    labs=modelo.fit_predict(Z)
    sil=silhouette_score(Z,labs,sample_size=min(1100,len(Z)),random_state=SEMENTE)
    estabilidade=[]
    for seed in (7,21,2026):
        alt=KMeans(n_clusters=k,n_init=10,random_state=seed).fit_predict(Z)
        estabilidade.append(adjusted_rand_score(labs,alt))
    linhas.append({'k':k,'silhouette':sil,'ARI_estabilidade_media':np.mean(estabilidade),
                   'menor_cluster':int(np.bincount(labs).min()),'inercia':modelo.inertia_})
selecao_k=pd.DataFrame(linhas)
display(selecao_k.round(3))
selecao_k.to_csv(OUT/'03_diagnostico_k_clusters.csv',index=False)

# Avoid very small clusters (<5% of the sample) whenever possible.
val=selecao_k[selecao_k['menor_cluster']>=max(15,round(.05*len(Z)))]
if len(val)==0: val=selecao_k
K_ESCOLHIDO=int(val.sort_values(['silhouette','ARI_estabilidade_media'],ascending=False).iloc[0]['k'])
# Change only with a scientific justification documented in the manuscript.
print('Selected K (exploratory criterion):',K_ESCOLHIDO)

fig,ax=plt.subplots(figsize=(7,4))
ax.plot(selecao_k.k,selecao_k.silhouette,'o-',label='Silhouette')
ax.plot(selecao_k.k,selecao_k.ARI_estabilidade_media,'s--',label='ARI across random seeds')
ax.axvline(K_ESCOLHIDO,color='grey',linestyle=':',label='Selected K')
ax.set(xlabel='Number of clusters (K)',ylabel='Index (higher is better)',title='Cluster-count diagnostic')
ax.legend();plt.tight_layout()
plt.savefig(OUT/'figura_02_selecao_k.png',bbox_inches='tight');plt.show()

modelo_final=KMeans(n_clusters=K_ESCOLHIDO,n_init=30,random_state=SEMENTE)
rotulos=modelo_final.fit_predict(Z)
base['cluster']=rotulos+1
base['cluster_nome']=base['cluster'].map(lambda x:f'Cluster {x}')
PALETA={i+1:CORES[i] for i in range(K_ESCOLHIDO)}
print('Cluster sizes:');display(base['cluster'].value_counts().sort_index().to_frame('hotels'))

## 5. Interpretable Cluster Profiles

Each cluster should be described using **observed dimensions**, rather than solely the number assigned by K-Means. We calculate medians, quartiles, standardised means, and a comparative table of values across clusters.

Group names are proposed according to the dimension whose standardised mean exceeds the overall mean by the greatest margin; the researcher should review these names before publication. These names **do not imply causality**, the actual quality of services, or productivity.


In [ ]:
perfil_mediana=base.groupby('cluster')[NOMES].median()
perfil_q25=base.groupby('cluster')[NOMES].quantile(.25)
perfil_q75=base.groupby('cluster')[NOMES].quantile(.75)
perfil_mediana.rename(columns=DIMENSOES).to_csv(OUT/'04_perfis_mediana_clusters.csv')
perfil_q25.rename(columns=DIMENSOES).to_csv(OUT/'05_perfis_q25_clusters.csv')
perfil_q75.rename(columns=DIMENSOES).to_csv(OUT/'06_perfis_q75_clusters.csv')
med_z=pd.DataFrame(Z,columns=NOMES,index=base.index).assign(cluster=base.cluster.values).groupby('cluster')[NOMES].mean()

rotulos_texto={}
for cluster in sorted(base.cluster.unique()):
    ranking=med_z.loc[cluster].sort_values(ascending=False)
    maior=ranking.index[0]
    rotulos_texto[cluster]=(f'Cluster {cluster} | relatively low accessibility'
          if float(ranking.iloc[0])<0.20 else f'Cluster {cluster} | salient feature: {DIMENSOES[maior]}')
base['perfil_nome']=base.cluster.map(rotulos_texto)
print('Initial interpretations (to be reviewed by the author):')
for k,n in rotulos_texto.items():print(' ',n)
display(perfil_mediana.rename(columns=DIMENSOES).round(3))

fig,ax=plt.subplots(figsize=(11,4.6))
sns.heatmap(med_z.rename(columns=DIMENSOES),annot=True,fmt='.2f',cmap='RdBu_r',center=0,
            ax=ax,cbar_kws={'label':'Standardised mean within cluster (z)'})
ax.set(xlabel='Accessibility dimensions',ylabel='Grupo',title='Spatial profiles: above/below the overall mean')
ax.set_xticklabels(ax.get_xticklabels(),rotation=23,ha='right')
plt.tight_layout();plt.savefig(OUT/'figura_03_heatmap_perfis.png',bbox_inches='tight');plt.show()

In [ ]:
# Histograms of the four features BY CLUSTER (shared scale within each row)
fig,axs=plt.subplots(len(NOMES),K_ESCOLHIDO,figsize=(3.4*K_ESCOLHIDO,2.6*len(NOMES)),squeeze=False,sharex='row')
for linha,col in enumerate(NOMES):
    x_all=base[col].dropna()
    limites=np.quantile(x_all,[0,.99]) if len(x_all)>5 else [0,1]
    if limites[0]==limites[1]:limites[1]+=1e-6
    for j,cluster in enumerate(sorted(base.cluster.unique())):
        ax=axs[linha,j]
        valores=base.loc[base.cluster==cluster,col].dropna()
        bins_comuns=np.linspace(limites[0],limites[1],26)
        ax.hist(valores.clip(limites[0],limites[1]),bins=bins_comuns,color=PALETA[cluster],alpha=.84)
        ax.set_xlim(limites[0],limites[1])
        ax.axvline(valores.median(),color='black',ls='--',lw=1,label='Median')
        ax.set_title(f'Cluster {cluster} | n={len(valores)}',fontsize=10)
        if j==0:ax.set_ylabel(DIMENSOES[col])
        if linha==len(NOMES)-1:ax.set_xlabel('Index / log-density')
        if j>0:ax.set_yticklabels([])
fig.suptitle('Spatial feature distributions by cluster (visual clipping at p99)',y=1.01)
plt.tight_layout();plt.savefig(OUT/'figura_04_histogramas_clusters_dimensoes.png',bbox_inches='tight');plt.show()

# Radar chart of normalised profiles; complements the heatmap, but does not replace numerical tables.
from matplotlib.ticker import MaxNLocator
valores_radar=perfil_mediana.copy()
for col in valores_radar:
    mn,mx=valores_radar[col].min(),valores_radar[col].max()
    valores_radar[col]=(valores_radar[col]-mn)/(mx-mn) if mx>mn else 0.5
ang=np.linspace(0,2*np.pi,len(NOMES),endpoint=False)
ang_f=np.r_[ang,ang[0]]
fig=plt.figure(figsize=(7.5,6.8))
ax=fig.add_subplot(111,polar=True)
for cluster,row in valores_radar.iterrows():
    v=np.r_[row.values,row.values[0]]
    ax.plot(ang_f,v,lw=2,color=PALETA[cluster],label=f'Cluster {cluster}')
    ax.fill(ang_f,v,alpha=.07,color=PALETA[cluster])
ax.set_xticks(ang);ax.set_xticklabels(list(DIMENSOES.values()),fontsize=8)
ax.set_ylim(0,1.05);ax.set_title('Radar chart: group-wise normalised medians',pad=25)
ax.legend(loc='upper right',bbox_to_anchor=(1.40,1.25));plt.tight_layout()
plt.savefig(OUT/'figura_05_radar_perfis.png',bbox_inches='tight');plt.show()

## 6. Maps: Clusters and Service Density

The maps assign a colour to each **spatial profile**. These are descriptive representations: belonging to a spatial group does not establish an effect on productivity. We save one interactive HTML map covering Portugal, including the islands, and another focused on mainland Portugal. The maps use online *tiles* when Internet access is available, but the point coordinates and colours are embedded in the HTML.

We avoid constructing a misleading continuous productivity surface: the points indicate **hotel locations**, not all buildings or residents of the municipalities.


In [ ]:
# Static maps (do not require Internet access or external services).
fig,axs=plt.subplots(1,2,figsize=(13,6.2))
for cluster,g in base.groupby('cluster'):
    axs[0].scatter(g.ht_longitude,g.ht_latitude,c=PALETA[cluster],s=12,alpha=.60,
                   label=f'Cluster {cluster} (n={len(g)})',rasterized=True)
continente=base[base.ht_longitude.between(-10.2,-6.0) & base.ht_latitude.between(36.5,42.4)]
for cluster,g in continente.groupby('cluster'):
    axs[1].scatter(g.ht_longitude,g.ht_latitude,c=PALETA[cluster],s=15,alpha=.70,
                   label=f'Cluster {cluster}',rasterized=True)
axs[0].set(xlabel='Longitude',ylabel='Latitude',title='Portugal and islands: spatial profiles')
axs[1].set(xlabel='Longitude',ylabel='Latitude',title='Mainland Portugal: spatial profiles')
axs[0].legend(loc='lower left',fontsize=8,frameon=True)
for ax in axs: ax.set_aspect('equal',adjustable='datalim')
plt.tight_layout();plt.savefig(OUT/'figura_06_mapa_clusters_estatico.png',bbox_inches='tight');plt.show()

# Do not include tax information or personal contact details in map pop-ups.
if FOLIUM_OK:
    def create_cluster_map(dados,ficheiro,centro,zoom):
        m=folium.Map(location=centro,zoom_start=zoom,tiles='CartoDB positron',control_scale=True)
        for cl in sorted(dados.cluster.unique()):
            grp=folium.FeatureGroup(name=f'Cluster {cl}: {len(dados[dados.cluster==cl])} hotels',show=True)
            for r in dados.loc[dados.cluster==cl].itertuples(index=False):
                muni=html.escape(str(getattr(r,'ht_municipality','')))
                pop=(f'<b>Hotel reg. {html.escape(str(r.hotel_key))}</b>'
                     f'<br>Municipality: {muni}'
                     f'<br>Cluster: {int(cl)}'
                     f'<br>Restaurants/cafés within 800 m: {getattr(r,"n_restaurantes_r0p8km",float("nan")):.0f}')
                folium.CircleMarker(location=[r.ht_latitude,r.ht_longitude],radius=4,
                   color=PALETA[cl],fill=True,fill_color=PALETA[cl],fill_opacity=.75,
                   weight=1,popup=folium.Popup(pop,max_width=300)).add_to(grp)
            grp.add_to(m)
        # Additional layer: castle POIs available from the cross join.
        ficheiro_pois=OUT/'13_pontos_castelos.csv'
        if ficheiro_pois.exists():
            pontos=pd.read_csv(ficheiro_pois)
            if 0<len(pontos)<=1500:
                layer=folium.FeatureGroup(name=f'Castle POIs ({len(pontos)})',show=False)
                for r in pontos.itertuples(index=False):
                    folium.CircleMarker([r.latitude,r.longitude],radius=3,color='#333333',
                       fill=True,fill_color='#333333',fill_opacity=.65,weight=1,
                       tooltip=html.escape(str(r.nome))).add_to(layer)
                layer.add_to(m)
        folium.LayerControl(collapsed=False).add_to(m)
        m.fit_bounds([[float(dados.ht_latitude.min()),float(dados.ht_longitude.min())],
                      [float(dados.ht_latitude.max()),float(dados.ht_longitude.max())]])
        m.save(str(OUT/ficheiro))
    create_cluster_map(base,'mapa_interativo_clusters_portugal.html',[39.4,-12.5],6)
    create_cluster_map(continente,'mapa_interativo_clusters_continente.html',[39.7,-8.0],7)
    print('HTML maps generated. Open them in your browser:')
    print(' ',(OUT/'mapa_interativo_clusters_portugal.html').name)
    print(' ',(OUT/'mapa_interativo_clusters_continente.html').name)
else:
    print('Folium is not installed: static plots are available; install folium to generate HTML maps.')

In [ ]:
# Restaurant/café density map within 800 m (not population density).
col='n_restaurantes_r0p8km'
fig,axs=plt.subplots(1,2,figsize=(13,5.5))
plot=continente.copy()
lims=plot[col].quantile([.02,.98]).values
lims = plot[col].quantile([.02, .98]).to_numpy(copy=True)
sc=axs[0].scatter(plot.ht_longitude,plot.ht_latitude,
                  c=plot[col].clip(lims[0],lims[1]),s=18,alpha=.88,
                  cmap='YlOrRd',vmin=lims[0],vmax=lims[1],rasterized=True)
axs[0].set(xlabel='Longitude',ylabel='Latitude',title='Restaurants/cafés within an 800 m radius')
cb=fig.colorbar(sc,ax=axs[0],fraction=.04,pad=.025);cb.set_label('Count (visually clipped at p98)')
sns.boxplot(data=base,x='cluster',y=col,hue='cluster',palette=PALETA,
            legend=False,showfliers=False,ax=axs[1])
axs[1].set(xlabel='Cluster',ylabel='Restaurants/cafés within 800 m',title='Local service density by group (visual outliers omitted)')
plt.tight_layout();plt.savefig(OUT/'figura_07_densidade_restaurantes.png',bbox_inches='tight');plt.show()
if FOLIUM_OK:
    nums=base['n_restaurantes_r0p8km'].fillna(0)
    max_c=float(max(1,nums.quantile(.98)))
    cores_dens=linear.YlOrRd_09.scale(0,max_c)
    m=folium.Map(location=[39.7,-8.1],zoom_start=7,tiles='CartoDB positron')
    for r in continente.itertuples(index=False):
        v=getattr(r,'n_restaurantes_r0p8km',float('nan'))
        if pd.isna(v):continue
        color=cores_dens(min(max_c,float(v)))
        folium.CircleMarker([r.ht_latitude,r.ht_longitude],radius=3,
             color=color,fill=True,fill_color=color,fill_opacity=.75,weight=1,
             tooltip=f'{int(v)} restaurants/cafés (800 m)').add_to(m)
    cores_dens.caption='Restaurants and cafés within 800 m (scale capped at p98)'
    cores_dens.add_to(m)
    m.save(str(OUT/'mapa_densidade_restaurantes_continente.html'))

## 7. External Interpretation: Hotel Structure, Geography, and Productivity

**Methodological rule:** The clusters are formed exclusively from the four spatial dimensions. Variables such as hotel star rating, capacity, municipality, and productivity may be used to **describe** the groups afterwards, but do not determine group membership.

The 2024 productivity analysis considers only observations from that year for which the target is available; the hotel remains the unit of analysis. Establishing whether a profile affects productivity would require controlling for other factors, historically validating the POI data, and conducting dedicated predictive/statistical analyses.


In [ ]:
# Join 2024 productivity from the panel for descriptive analysis only; not used in K-Means.
observado_2024=painel[painel.f_year.eq(2024)][['hotel_key','f_labprodct']].copy()
observado_2024=observado_2024.drop_duplicates('hotel_key')
base=base.merge(observado_2024,on='hotel_key',how='left',validate='one_to_one')
print('Hotels with observed productivity in 2024:',base.f_labprodct.notna().sum())
cols_perfil=['ht_capacity','ht_quality_stars','n_restaurantes_r0p8km','f_labprodct']
resumo_cluster=base.groupby('cluster').agg(
    n_hoteis=('hotel_key','nunique'),
    capacidade_mediana=('ht_capacity','median'),
    estrelas_mediana=('ht_quality_stars','median'),
    servicos_800m_mediana=('n_restaurantes_r0p8km','median'),
    produtividade_2024_n=('f_labprodct','count'),
    produtividade_2024_mediana=('f_labprodct','median'),
    produtividade_2024_media=('f_labprodct','mean')
)
resumo_cluster.to_csv(OUT/'07_descricao_externa_clusters.csv')
display(resumo_cluster.round(2))

fig,axs=plt.subplots(1,2,figsize=(12,4.4))
sns.boxplot(data=base,x='cluster',y='ht_capacity',hue='cluster',palette=PALETA,
            legend=False,showfliers=False,ax=axs[0])
axs[0].set(title='Hotel capacity by spatial profile',xlabel='Cluster',ylabel='Capacidade')
prod=base.dropna(subset=['f_labprodct']).copy()
if len(prod):
    corte=float(prod.f_labprodct.quantile(.99))
    prod['prod_visual']=prod.f_labprodct.clip(upper=corte)
    sns.boxplot(data=prod,x='cluster',y='prod_visual',hue='cluster',palette=PALETA,
                legend=False,showfliers=False,ax=axs[1])
    axs[1].set(title='2024 productivity by cluster (visual clipping at p99)',xlabel='Cluster',ylabel='Productivity (original units)')
else:
    axs[1].text(.5,.5,'No valid 2024 productivity observations',ha='center',va='center')
plt.tight_layout();plt.savefig(OUT/'figura_08_caracterizacao_produtividade.png',bbox_inches='tight');plt.show()

# NUTS regional cross-tabulation to identify geographically driven clusters.
cross=pd.crosstab(base['ht_nuts_2_newcriteria'].fillna('Unknown region'),base['cluster'])
cross.to_csv(OUT/'08_regioes_x_cluster_contagens.csv')
regi=cross.loc[cross.sum(axis=1).sort_values(ascending=False).head(12).index]
fig,ax=plt.subplots(figsize=(10,5))
regi.div(regi.sum(axis=1),axis=0).plot.barh(stacked=True,color=[PALETA[int(k)] for k in regi.columns],ax=ax)
ax.set(title='Profile composition by region (12 most represented regions)',xlabel='Fraction of hotels',ylabel='Region')
ax.legend(title='Cluster',bbox_to_anchor=(1.02,1),loc='upper left')
plt.tight_layout();plt.savefig(OUT/'figura_09_composicao_regional.png',bbox_inches='tight');plt.show()

In [ ]:
# Two-dimensional PCA display using ONLY the four variables used for clustering.
pca=PCA(n_components=2,random_state=SEMENTE)
xy=pca.fit_transform(Z)
fig,ax=plt.subplots(figsize=(9,6))
for k in sorted(base.cluster.unique()):
    sub=xy[base.cluster.to_numpy()==k]
    ax.scatter(sub[:,0],sub[:,1],s=18,alpha=.70,c=PALETA[k],label=f'Cluster {k}',rasterized=True)
ax.set(xlabel=f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)',
       ylabel=f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)',
       title='PCA representation of spatial profiles')
ax.legend();plt.tight_layout()
plt.savefig(OUT/'figura_10_pca_clusters.png',bbox_inches='tight');plt.show()
print('Explained variance of the first two components:',round(float(pca.explained_variance_ratio_.sum()),3))

## 8. Outputs for the Paper and Integration with the Panel

Each hotel is assigned **one** cluster and a set of spatial indicators. To link these assignments to the annual 2018–2024 panel, we use `hotel_key` (registration number) rather than joining the Cartesian table directly to every annual observation. This prevents artificial duplication of rows and distortion of model training and statistical tests.

**Caveats for submission:**

- The temporal provenance of POIs must be documented. A map of currently observed points **does not automatically represent** the supply available in 2018.
- Distances are straight-line (Haversine) distances, **not** road distances or travel times. Geographic differences between mainland Portugal and the islands must be examined.
- K-Means with standardised features and *silhouette*-based $K$ selection yields an *exploratory* taxonomy. It does not establish natural groups or causal effects.
- Aggregation into $A(k)$ and $D(r)$ depends on POI coverage and cataloguing procedures. Castles are relatively sparse; restaurants/cafés have much higher densities.
- If clusters are subsequently used as regression *features*, all fitted transformations must be learned **within each temporal fold**, without using test observations to learn the transformation. This notebook provides descriptive EDA, not a predictive benchmark.


In [ ]:
# Produce final artefacts for the Data / Methodology section.
para_exportar=['hotel_key','ht_id','ht_latitude','ht_longitude','ht_municipality',
 'ht_nuts_2_newcriteria','ht_quality_stars','ht_capacity',
 'cluster','perfil_nome','n_dimensoes_validas']
para_exportar += [c for c in base.columns if c.startswith(('a_','d_castelos_','d_igrejas_','d_praias_',
                                     'n_castelos_','n_igrejas_','n_praias_','n_restaurantes_',
                                     'dens_','log_dens_'))]
para_exportar=list(dict.fromkeys(c for c in para_exportar if c in base))
export_hotel=base[para_exportar].copy()
export_hotel.to_csv(OUT/'09_indicadores_e_clusters_por_hotel.csv',index=False)

# Associate one spatial record per hotel with the annual panel, without duplicating panel rows.
painel_export=painel.merge(
    export_hotel[['hotel_key','cluster']+NOMES],on='hotel_key',how='left',validate='many_to_one'
)
assert len(painel_export)==len(painel),'ERROR: the spatial merge duplicated panel rows!'
assert not export_hotel.duplicated('hotel_key').any()
painel_export.to_csv(OUT/'10_painel_com_indicadores_espaciais.csv',index=False)

metadata={
 'ano_auditoria':'2026',
 'observacoes_painel':int(len(painel)),
 'hoteis_unicos_espaciais':int(len(base)),
 'k_clusters':int(K_ESCOLHIDO),
 'dimensoes_clustering':NOMES,
 'k_vizinhos_clustering':int(K_ACESSO_CLUSTER),
 'raios_km':RAIOS_KM,
 'km_referencia_indice':D0_KM,
 'fontes_por_categoria':fontes,
 'criterio_clustering':'KMeans / silhouette / clusters >=5%, excluding productivity',
 'geodesico':'Haversine (R=6371.0088km)',
 'recolha_historica_poi':'NOT DOCUMENTED: confirm with data author before submission',
 'cautela':'Descriptive profiles; no causal claim or predictive training with clusters fitted outside folds.'
}
(OUT/'11_metadados_metodologia.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print(f'Exported {len(export_hotel)} hotels and {len(painel_export)} panel rows.')
print('Files created:')
for caminho in sorted(OUT.iterdir()):
    if caminho.is_file():print('  ',caminho.name)

## 9. Candidate Figures and Tables for the Manuscript

**Figure A.** *Spatial distribution of independent hotels and unsupervised tourism-accessibility profiles* (cluster-coloured map; include mainland Portugal and the islands, using a consistent palette).

**Figure B.** *Tourism-accessibility dimensions distinguishing the hotel clusters* (standardised heatmap accompanied by a table in the original units).

**Figure C.** *Distribution of tourism proximity and local service intensity across spatial clusters* (histograms by dimension and group).

**Figure D.** *Hotel characteristics and labour productivity by tourism-accessibility profiles* (descriptive analysis, avoiding causal language).

**Table A.** *Construction and provenance of spatial features*: category, geographic source, number of POIs, distance measures in km, counts within 0.8/2/5/10 km, scale, and data-collection date.

**Table B.** *Cluster profiles*: $N$, access to castles/churches/beaches, restaurants and cafés, capacity, star rating, and median productivity in 2024; spatial dimensions are computed without using the productivity variable.

**Question for further research:** Do spatial-context clusters add predictive value beyond structural attributes and temporal memory? To answer this, fit the transformations on the training portion of each benchmark fold and compare models on the same observations and years.
